# Notebook 06 — Genetic Algorithm Optimizer

**Green Slotting: Phase 2 of the Optimization Stage**

This notebook assigns products to warehouse slots using a genetic algorithm, then
runs it twice for a robustness check: once fed LightGBM demand forecasts (**GA-ML**),
once fed Ridge regression forecasts (**GA-Ridge**). If GA-ML produces a better layout,
that's evidence the better forecast really does translate into a better warehouse.

### The model: replicated stock (not one-slot-per-product)

Real warehouses stock popular items in *several* locations so a picker is never far
from one. The baseline files in this project confirm it: each physical location holds
up to **18 products**, and each product typically sits in **~6 locations** (up to 28
for the most popular). We match that here rather than assuming one slot per product —
partly for realism, and partly because it's a hard requirement: 2,456 products simply
don't fit one-each into 2,292 locations.

Two data-driven corrections worth knowing before you run this:

1. **Replica counts are capped and demand-compressed, not linear.** A linear
   demand-proportional split would hand the single top-selling shoe over 500 slots
   (some products are ~140x more popular than the median). Real warehouses don't do
   that -- they cap replication and grow it sub-linearly with demand. We use `sqrt(demand)`
   as the apportionment basis with a cap of 28 replicas, which reproduces the real
   baseline's mean (~6) and max (~28) closely.
2. **We only place the ~2,456 products we have forecasts for**, using a realistic
   replica budget (not the full theoretical warehouse capacity of 41,256 units -- the
   real warehouse stocks ~7,163 distinct SKUs total, most outside our order-history
   window, so forcing our smaller product set to fill *all* capacity would over-replicate
   everything). The unused capacity is implicitly reserved for products outside this
   study's scope.

### Why a GA is needed at all

Minimizing weighted distance to the dock *alone* has a closed-form optimal solution
(sort products by demand, sort slots by distance, pair them up -- the rearrangement
inequality). No search required, and no role for a GA. What makes this a genuine
optimization problem is a second objective pulling in a different direction:
**co-picked products should sit near each other**, not just near the dock. We mine
real co-picking pairs from `Picking_Wave.csv` (unsurprisingly, the strongest pairs
turn out to be the same shoe model in adjacent sizes) and add that as a competing
term in the fitness function. That's what makes the problem non-separable and gives
the GA something real to search for.

**Inputs** (same folder as this notebook):
- `model_results.csv` -- LightGBM demand forecasts
- `baseline_results.csv` -- Ridge regression forecasts (for the robustness check)
- `distance_lookup.csv` -- from Notebook 05
- `Picking_Wave.csv` -- real picking history (for co-picking affinity)

**Outputs:**
- `ga_solution_ml.csv` -- best layout found using LightGBM demand
- `ga_solution_ridge.csv` -- best layout found using Ridge demand (robustness check)
- `ga_convergence.png` -- fitness over generations for both runs

Expect this notebook to run in well under a minute on a normal laptop.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from itertools import combinations
from collections import Counter, defaultdict

rng = np.random.default_rng(42)
print("Libraries loaded.")

In [ ]:
import os as _os
# ── Image output folder ─────────────────────────────────────────────────────
# Figures are saved to a "figures/" subfolder in the directory where you run
# this notebook. Change FIGURES_DIR below if you prefer a different path.
FIGURES_DIR = _os.path.join(_os.getcwd(), "figures")
_os.makedirs(FIGURES_DIR, exist_ok=True)
print(f"[INFO] Figures will be saved to: {FIGURES_DIR}")

## 2. Configuration

- **`CAPACITY_PER_LOCATION`** -- confirmed from the baseline files: each physical
  location holds up to 18 products.
- **`TARGET_MEAN_REPLICAS` / `MAX_REPLICAS`** -- calibrated to match the real
  baseline data (mean 5.88, max 28 observed in `Class_Based_Storage.csv`).
- **`LAMBDA_AFFINITY`** -- how much weight the co-picking term gets relative to the
  distance term, *after* both are put on a comparable scale (done automatically in
  Section 6 -- the two objectives differ by ~500x in raw magnitude, so combining them
  without normalizing would make the affinity term invisible to the GA).
- **`TOP_K_PAIRS`** -- number of strongest co-picking pairs to optimize for.

In [ ]:
DATA_DIR = "."

CAPACITY_PER_LOCATION = 18
TARGET_MEAN_REPLICAS = 6      # matches real baseline average (5.88)
MAX_REPLICAS = 28             # matches real baseline max
LAMBDA_AFFINITY = 0.15        # relative weight of co-picking term (post-normalization)
TOP_K_PAIRS = 800             # number of co-picking pairs to include in fitness

POP_SIZE = 120
N_GENERATIONS = 800
ELITE_FRAC = 0.10
TOURNAMENT_SIZE = 5
MUTATIONS_PER_INDIV = 25      # random swap mutations per individual per generation
N_TARGETED_PER_INDIV = 8      # affinity-directed "co-locate" mutations per individual
EARLY_STOP_WINDOW = 100
EARLY_STOP_THRESHOLD = 0.0001

print("Config set.")

## 3. Load locations & distances (from Notebook 05)

In [ ]:
loc_df = pd.read_csv(f"{DATA_DIR}/distance_lookup.csv")
n_locations = len(loc_df)
print(f"Loaded {n_locations} locations with distance-to-I/O.")
loc_df.head()

## 4. Hamilton apportionment (capped, demand-compressed)

Standard Hamilton/largest-remainder apportionment, extended with a per-product cap.
The weight fed in is `sqrt(demand)` rather than raw demand -- this compresses the huge
disparity between the top sellers and the long tail, which is what keeps replica
counts realistic instead of handing hundreds of slots to a handful of products.

In [ ]:
def hamilton_apportionment_capped(weights, total_seats, min_seats=1, max_seats=None):
    """Largest-remainder apportionment with a per-item cap. Every item gets >= min_seats."""
    weights = np.asarray(weights, dtype=float)
    n = len(weights)
    seats = np.full(n, min_seats)
    remaining = total_seats - n * min_seats
    active = np.ones(n, dtype=bool)
    for _ in range(50):
        w = weights * active
        if w.sum() == 0 or remaining <= 0:
            break
        quota = w / w.sum() * remaining
        base = np.floor(quota).astype(int)
        proposed = seats + base
        over = proposed > max_seats if max_seats else np.zeros(n, dtype=bool)
        if not over.any():
            rem = quota - base
            shortfall = remaining - base.sum()
            idx_order = np.argsort(-rem)
            for i in idx_order:
                if shortfall <= 0:
                    break
                if max_seats is None or proposed[i] < max_seats:
                    proposed[i] += 1
                    shortfall -= 1
            seats = proposed
            remaining = total_seats - seats.sum()
            break
        else:
            seats[over] = max_seats
            active = ~over & active
            remaining = total_seats - seats.sum()
    return seats

print("Apportionment function ready.")

## 5. Co-picking affinity pairs (from real picking waves)

`Picking_Wave.csv` records what was picked together in each wave. We count how often
every pair of products appears in the same wave, and keep the strongest pairs. This
only needs to be computed once -- both GA-ML and GA-Ridge share the same affinity
structure (it comes from real operations, not from either forecast).

In [ ]:
waves = pd.read_csv(f"{DATA_DIR}/Picking_Wave.csv", sep=";", encoding="utf-8-sig")
waves.columns = [c.strip() for c in waves.columns]
waves["product_id"] = waves["reference"].astype(str) + "_" + waves["Size (US)"].astype(str)

wave_groups = waves.groupby("waveNumber")["product_id"].apply(lambda x: sorted(set(x)))

pair_counts = Counter()
for products in wave_groups:
    if len(products) < 2:
        continue
    for a, b in combinations(products, 2):
        pair_counts[(a, b)] += 1

top_pairs = pair_counts.most_common(TOP_K_PAIRS)
print(f"Total unique co-picked pairs found: {len(pair_counts)}")
print(f"Using top {len(top_pairs)} by frequency.")
print("\nStrongest pairs (note: same model, adjacent sizes dominates -- expected):")
for (a, b), c in top_pairs[:5]:
    print(f"  {a} <-> {b}: picked together {c} times")

## 6. The GA engine (shared by both runs)

Everything below is packaged into one function, `run_ga(demand_df, label)`, so we can
call it once for LightGBM demand and once for Ridge demand with identical logic.

**Design notes:**
- The chromosome is a permutation of "capacity units" (physical slot positions,
  expanded so each location contributes 18 units). Fitness is fully vectorized with
  NumPy across the whole population, which is what keeps this fast despite the
  problem having ~15,000 replicas to place.
- We seed the initial population with the closed-form optimal (demand-sorted vs.
  distance-sorted) solution, then perturb it for diversity. This is the same
  "rearrangement inequality" solution that Dedicated Storage effectively uses -- the
  GA's job is to improve on it by incorporating the affinity term that pure sorting
  ignores.
- Mutation has two flavors: random swaps (broad exploration) and *targeted*
  co-location swaps (deliberately try to place a random affinity-paired product next
  to its partner). Pure random mutation turned out to almost never improve
  the affinity term in testing, because it's ~500x smaller in magnitude than the
  distance term -- targeted mutation is what actually makes the search work.

In [ ]:
def run_ga(demand_df, label, verbose=True):
    """
    demand_df: DataFrame with columns [product_id, demand]
    label: string used in print statements (e.g. 'ML', 'Ridge')
    Returns: solution_df, fitness_history (list of best fitness per generation)
    """
    t_start = time.time()
    demand_df = demand_df.copy()
    demand_df["demand"] = demand_df["demand"].clip(lower=0.001)
    n_products = len(demand_df)

    # --- replica counts ---
    target_total_replicas = round(TARGET_MEAN_REPLICAS * n_products)
    replica_basis = np.sqrt(demand_df["demand"].values)
    replica_counts = hamilton_apportionment_capped(
        replica_basis, target_total_replicas, min_seats=1, max_seats=MAX_REPLICAS)
    demand_df["replica_count"] = replica_counts
    if verbose:
        print(f"[{label}] Replica counts: mean={replica_counts.mean():.2f}, "
              f"max={replica_counts.max()}, sum={replica_counts.sum()}")

    # --- fixed arrays ---
    product_ids = demand_df["product_id"].values
    replica_products = np.repeat(np.arange(n_products), replica_counts)
    N = len(replica_products)

    demand_per_product = demand_df["demand"].values
    demand_share_by_product = demand_per_product / replica_counts
    demand_replica_arr = demand_share_by_product[replica_products]

    all_units = np.repeat(np.arange(n_locations), CAPACITY_PER_LOCATION)
    all_unit_distances = loc_df["dist_to_io"].values[all_units]
    closest_order = np.argsort(all_unit_distances)[:N]
    location_units = all_units[closest_order]
    rng.shuffle(location_units)
    distance_units = loc_df["dist_to_io"].values[location_units]

    if verbose:
        print(f"[{label}] Placing {N} replicas across {N} candidate capacity units.")

    first_replica_idx = np.full(n_products, -1)
    for k, p in enumerate(replica_products):
        if first_replica_idx[p] == -1:
            first_replica_idx[p] = k

    pid_to_idx = {pid: i for i, pid in enumerate(product_ids)}
    pair_a_idx, pair_b_idx, pair_w = [], [], []
    for (a, b), w in top_pairs:
        if a in pid_to_idx and b in pid_to_idx:
            pair_a_idx.append(pid_to_idx[a])
            pair_b_idx.append(pid_to_idx[b])
            pair_w.append(w)
    pair_a_idx = np.array(pair_a_idx)
    pair_b_idx = np.array(pair_b_idx)
    pair_w = np.array(pair_w, dtype=float)
    pair_w = pair_w / pair_w.max()
    rep_a = first_replica_idx[pair_a_idx]
    rep_b = first_replica_idx[pair_b_idx]
    n_pairs = len(pair_a_idx)
    loc_xy = loc_df[["x", "y"]].values

    location_to_unit_positions = defaultdict(list)
    for u_pos, loc_idx in enumerate(location_units):
        location_to_unit_positions[loc_idx].append(u_pos)
    location_to_unit_positions = {k: np.array(v) for k, v in location_to_unit_positions.items()}

    if verbose:
        print(f"[{label}] Using {n_pairs} affinity pairs for the co-picking term.")

    affinity_scale_holder = {"scale": 1.0}

    def evaluate_population(pop):
        dist_term = (demand_replica_arr[np.newaxis, :] * distance_units[pop]).sum(axis=1)

        unit_a = pop[:, rep_a]
        unit_b = pop[:, rep_b]
        loc_a = location_units[unit_a]
        loc_b = location_units[unit_b]
        xy_a = loc_xy[loc_a]
        xy_b = loc_xy[loc_b]
        manh = np.abs(xy_a[:, :, 0] - xy_b[:, :, 0]) + np.abs(xy_a[:, :, 1] - xy_b[:, :, 1])
        affinity_term_raw = (manh * pair_w[np.newaxis, :]).sum(axis=1)

        fitness = dist_term + LAMBDA_AFFINITY * affinity_term_raw * affinity_scale_holder["scale"]
        return fitness, dist_term, affinity_term_raw

    order_by_demand = np.argsort(-demand_replica_arr)
    order_by_distance = np.argsort(distance_units)
    seed = np.empty(N, dtype=int)
    seed[order_by_demand] = order_by_distance

    seed_fit, seed_dist, seed_aff = evaluate_population(seed[np.newaxis, :])
    affinity_scale_holder["scale"] = seed_dist[0] / max(seed_aff[0], 1e-9)
    if verbose:
        print(f"[{label}] Affinity scale calibration: dist={seed_dist[0]:,.0f}, "
              f"aff_raw={seed_aff[0]:,.0f}, scale={affinity_scale_holder['scale']:,.1f}")

    pop = np.zeros((POP_SIZE, N), dtype=int)
    pop[0] = seed
    for i in range(1, POP_SIZE):
        ind = seed.copy()
        n_perturb = rng.integers(50, 500)
        ia = rng.integers(0, N, n_perturb)
        ib = rng.integers(0, N, n_perturb)
        ind[ia], ind[ib] = ind[ib].copy(), ind[ia].copy()
        pop[i] = ind

    n_elite = max(1, int(POP_SIZE * ELITE_FRAC))
    fitness_history = []

    for gen in range(N_GENERATIONS):
        fitness, dist_term, aff_term = evaluate_population(pop)
        best_idx = np.argmin(fitness)
        fitness_history.append(fitness[best_idx])

        if gen > EARLY_STOP_WINDOW:
            recent = fitness_history[-EARLY_STOP_WINDOW:]
            if (recent[0] - recent[-1]) / recent[0] < EARLY_STOP_THRESHOLD:
                if verbose:
                    print(f"[{label}] Early stop at generation {gen} (plateau).")
                break

        new_pop = np.zeros_like(pop)
        elite_idx = np.argsort(fitness)[:n_elite]
        new_pop[:n_elite] = pop[elite_idx]

        for i in range(n_elite, POP_SIZE):
            contenders = rng.integers(0, POP_SIZE, TOURNAMENT_SIZE)
            winner = contenders[np.argmin(fitness[contenders])]
            child = pop[winner].copy()

            ia = rng.integers(0, N, MUTATIONS_PER_INDIV)
            ib = rng.integers(0, N, MUTATIONS_PER_INDIV)
            child[ia], child[ib] = child[ib].copy(), child[ia].copy()

            inv_child = np.empty(N, dtype=int)
            inv_child[child] = np.arange(N)

            pair_choices = rng.integers(0, n_pairs, N_TARGETED_PER_INDIV)
            for j in pair_choices:
                a_rep, b_rep = rep_a[j], rep_b[j]
                b_unit = child[b_rep]
                b_loc = location_units[b_unit]
                candidates = location_to_unit_positions.get(b_loc)
                if candidates is None or len(candidates) == 0:
                    continue
                target_unit = candidates[rng.integers(0, len(candidates))]
                occupant_replica = inv_child[target_unit]
                if occupant_replica == a_rep:
                    continue
                a_unit = child[a_rep]
                child[a_rep], child[occupant_replica] = child[occupant_replica], child[a_rep]
                inv_child[a_unit] = occupant_replica
                inv_child[target_unit] = a_rep

            new_pop[i] = child

        pop = new_pop

        if verbose and (gen % 50 == 0):
            print(f"[{label}] Gen {gen:4d} | best fitness = {fitness[best_idx]:,.0f} "
                  f"(dist={dist_term[best_idx]:,.0f}, aff={aff_term[best_idx]:,.0f})")

    final_fitness, final_dist, final_aff = evaluate_population(pop)
    best_idx = np.argmin(final_fitness)
    best_genome = pop[best_idx]

    rand_perm = rng.permutation(N)
    rand_fitness, _, _ = evaluate_population(rand_perm[np.newaxis, :])
    improvement = 100 * (rand_fitness[0] - final_fitness[best_idx]) / rand_fitness[0]

    elapsed = time.time() - t_start
    print(f"\n[{label}] DONE in {elapsed:.1f}s | "
          f"best fitness={final_fitness[best_idx]:,.0f} | "
          f"vs random improvement={improvement:.1f}%")

    solution_df = pd.DataFrame({
        "product_id": product_ids[replica_products],
        "demand_share": demand_replica_arr,
        "assigned_location": loc_df["location_id"].values[location_units[best_genome]],
        "distance_to_io": distance_units[best_genome],
    })
    solution_df["weighted_distance"] = solution_df["demand_share"] * solution_df["distance_to_io"]

    return solution_df, fitness_history

print("run_ga() defined.")

## 7. Run GA-ML (LightGBM demand)

In [ ]:
demand_ml = pd.read_csv(f"{DATA_DIR}/model_results.csv")[["unique_id", "lgb_pred"]].rename(
    columns={"unique_id": "product_id", "lgb_pred": "demand"})

solution_ml, history_ml = run_ga(demand_ml, label="ML")

## 8. Run GA-Ridge (Ridge regression demand) -- the robustness check

Same GA, same affinity pairs, same everything -- the only thing that changes is which
forecast supplies the demand weights. If GA-ML ends up with a lower fitness (shorter
weighted travel) than GA-Ridge, that's evidence the better forecast really does
produce a better warehouse layout, closing the "better forecast leads to better layout"
argument for the paper.

In [ ]:
demand_ridge = pd.read_csv(f"{DATA_DIR}/baseline_results.csv")[["unique_id", "ridge_pred"]].rename(
    columns={"unique_id": "product_id", "ridge_pred": "demand"})

solution_ridge, history_ridge = run_ga(demand_ridge, label="Ridge")

## 9. Compare GA-ML vs GA-Ridge

**Important caveat:** these two solutions place a *different number of total
replicas* (replica counts are computed independently per demand source, since the
two forecasts disagree on which products are "high demand"). So the raw weighted-distance
totals below aren't directly comparable -- a fair comparison needs both layouts
replayed against the same real picking waves, which happens in Notebook 07. What we
*can* compare here is the shape of convergence, and how each got demand-heavy
products close to the dock.

In [ ]:
fitness_ml = solution_ml["weighted_distance"].sum()
fitness_ridge = solution_ridge["weighted_distance"].sum()

print(f"GA-ML    total weighted distance: {fitness_ml:,.0f}  ({len(solution_ml)} replicas placed)")
print(f"GA-Ridge total weighted distance: {fitness_ridge:,.0f}  ({len(solution_ridge)} replicas placed)")

print("\nTop-5 demand products, GA-ML placement:")
top_ml = solution_ml.groupby("product_id").agg(
    replicas=("product_id","size"), mean_dist=("distance_to_io","mean")
).sort_values("mean_dist").head(5)
print(top_ml)

## 10. Convergence plot

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

ax[0].plot(history_ml, color="steelblue", linewidth=2)
ax[0].set_title("GA-ML convergence")
ax[0].set_xlabel("generation"); ax[0].set_ylabel("best fitness")
ax[0].grid(alpha=0.3)

ax[1].plot(history_ridge, color="darkorange", linewidth=2)
ax[1].set_title("GA-Ridge convergence")
ax[1].set_xlabel("generation"); ax[1].set_ylabel("best fitness")
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(_os.path.join(FIGURES_DIR, "06_ga_convergence.png"), dpi=150, bbox_inches="tight")
plt.show()

## 11. Save outputs

These two files are the handoff to Notebook 07, which replays real picking waves
against both GA layouts and the four baseline strategies (Random, Dedicated,
Class-Based, Hybrid) for the ground-truth comparison.

In [ ]:
solution_ml.to_csv("ga_solution_ml.csv", index=False)
solution_ridge.to_csv("ga_solution_ridge.csv", index=False)

print("Saved:")
print("  ga_solution_ml.csv     -- GA layout using LightGBM demand")
print("  ga_solution_ridge.csv  -- GA layout using Ridge demand (robustness check)")
print("  ga_convergence.png     -- convergence plots for both runs")

## Done -- Phase 2 complete

Both GA variants have found layouts. The surrogate fitness used here (distance +
co-picking affinity) is a *search signal*, not the final answer -- the real test is
how these layouts perform against actual picking waves, which is what Notebook 07
does next, alongside the four baseline strategies already in this project
(Random, Dedicated, Class-Based, Hybrid).

In [ ]:
# ── Extra graphic: Demand vs replicas + placement quality heatmap ──────────
import matplotlib.pyplot as plt
import numpy as np
import os as _os

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Demand vs replica count (ML solution)
rep_summary = solution_ml.groupby("product_id").agg(
    replicas=("product_id", "count"),
    demand=("demand_share", "sum"),
    mean_dist=("distance_to_io", "mean")
).reset_index()

sc = axes[0].scatter(rep_summary["demand"], rep_summary["replicas"],
                     c=rep_summary["mean_dist"], cmap="RdYlGn_r",
                     alpha=0.6, s=20, edgecolors="none")
plt.colorbar(sc, ax=axes[0], label="Mean dist to I/O")
axes[0].set_xlabel("Total demand (LightGBM forecast)")
axes[0].set_ylabel("Replicas assigned")
axes[0].set_title("GA-ML: Demand vs Replicas\n(colour = proximity to dock)", fontweight="bold")
axes[0].grid(alpha=0.2)

# GA-ML vs GA-Ridge: mean distance per product (scatter)
ml_dist = solution_ml.groupby("product_id")["distance_to_io"].mean().rename("ml_dist")
ridge_dist = solution_ridge.groupby("product_id")["distance_to_io"].mean().rename("ridge_dist")
comp = pd.concat([ml_dist, ridge_dist], axis=1, join="inner")
axes[1].scatter(comp["ml_dist"], comp["ridge_dist"], alpha=0.4, s=10, color="#4C72B0")
lim_lo = min(comp["ml_dist"].min(), comp["ridge_dist"].min()) * 0.98
lim_hi = max(comp["ml_dist"].max(), comp["ridge_dist"].max()) * 1.02
axes[1].plot([lim_lo, lim_hi], [lim_lo, lim_hi], "k--", linewidth=1, label="Equal distance")
axes[1].set_xlabel("GA-ML: mean distance to I/O per product")
axes[1].set_ylabel("GA-Ridge: mean distance to I/O per product")
axes[1].set_title("GA-ML vs GA-Ridge\nPer-product placement proximity", fontweight="bold")
axes[1].legend(); axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig(_os.path.join(FIGURES_DIR, "06_placement_analysis.png"), dpi=150, bbox_inches="tight")
plt.show()